<a href="https://colab.research.google.com/github/leoncana/CreativeHub-DataEngineeringModule1/blob/main/notebooks/session04_spark_transformations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 4 — Spark part 2: schemas, filters and data quality

Last session we started Spark, built our first DataFrames and calculated new columns with `F.col` and functions.
Today we continue with the tools you need every day: telling Spark the types, keeping only the rows you want, sorting —
and then we use Spark to **investigate** the messy listening log from session 2.

**Today you will learn**
1. Schemas: telling Spark the types of the columns
2. The core transformations: `filter`, `orderBy`, `distinct`, combining conditions
3. Lazy evaluation: transformations vs actions
4. Using Spark to **investigate** data quality: nulls, duplicates, `groupBy().count()`, `F.when`

## Homework review (15 min)
Open `homework/hw03_spark_first_steps.ipynb`.

---
## Setup
Run both cells. The second one starts Spark — it takes a few seconds.

In [1]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

Course data: /content/course/data


In [2]:
# Start Spark. Colab already has PySpark; anywhere else we install it first.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pyspark>=3.5,<3.6"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable  # Spark's workers must use the same Python as the notebook

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("wavely")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")  # same behaviour on Spark 3.5 and 4.x: bad casts give null
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready")

Spark 4.0.5 is ready


---
## Recap — session 3 in one cell

Run it and say out loud what each line does. Everything here is from last session.

In [3]:
tracks_guessed = spark.read.csv(str(DATA / "reference" / "tracks.csv"), header=True, inferSchema=True)

(
    tracks_guessed
    .withColumn("minutes", F.round(F.col("duration_ms") / 60000, 1))
    .withColumn("title_upper", F.upper(F.col("title")))
    .select("title", "title_upper", "genre", "minutes")
    .show(3)
)

+----------------+----------------+-----+-------+
|           title|     title_upper|genre|minutes|
+----------------+----------------+-----+-------+
|Electric Mirrors|ELECTRIC MIRRORS|  pop|    4.0|
|Electric Skyline|ELECTRIC SKYLINE|  pop|    3.3|
|Endless Promises|ENDLESS PROMISES|  pop|    3.3|
+----------------+----------------+-----+-------+
only showing top 3 rows


---
## Part 1 — Schemas: telling Spark the types

Let's read the messy listening log from session 2, this time without `inferSchema`:

In [4]:
log = spark.read.csv(str(DATA / "messy" / "listening_log_messy.csv"), header=True)

log.show(5)
log.printSchema()
print(log.count(), "rows")

+-------+-------------------+-----+--------------+-------------------+--------+-------+------+
|play_id|          played_at| user|        artist|              track|duration|country|device|
+-------+-------------------+-----+--------------+-------------------+--------+-------+------+
|      1|   01/09/2026 23:56|U0293| Calvin Harris|  Little Promises  |      47|     DE|mobile|
|      2|   03/09/2026 21:21|U0025| Billie Eilish|    Little Lights  |    1:25|     AT|Mobile|
|      3|2026-09-02 12:52:37|U0075|        EMINEM| Northern Thunder  |      86|     SK|mobile|
|      4|   02/09/2026 18:38|U0046|        Eminem|     Broken Shadows|    0:17|     SK|mobile|
|      5|2026-09-06 11:03:45|U0396|Kendrick Lamar|    electric lights|      87|     AT|Mobile|
+-------+-------------------+-----+--------------+-------------------+--------+-------+------+
only showing top 5 rows
root
 |-- play_id: string (nullable = true)
 |-- played_at: string (nullable = true)
 |-- user: string (nullable = true)


Every column is a `string`. CSV has no types (remember homework 2?). Two ways to get types:

1. `inferSchema=True` — Spark reads the data an extra time and guesses. Handy for exploring, slow and risky for production:
   if one day a column contains a bad value, the guessed type silently changes.
2. An explicit **schema** — you tell Spark the types. This is what data engineers do in pipelines.

The easiest way to write a schema is a string `"column TYPE, column TYPE, ..."`. The most common types are
`STRING`, `INT`, `BIGINT`, `DOUBLE`, `BOOLEAN`, `DATE` and `TIMESTAMP`:

In [5]:
log_typed = spark.read.csv(
    str(DATA / "messy" / "listening_log_messy.csv"),
    header=True,
    schema="play_id INT, played_at STRING, user STRING, artist STRING, track STRING, duration STRING, country STRING, device STRING",
)
log_typed.printSchema()

root
 |-- play_id: integer (nullable = true)
 |-- played_at: string (nullable = true)
 |-- user: string (nullable = true)
 |-- artist: string (nullable = true)
 |-- track: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- country: string (nullable = true)
 |-- device: string (nullable = true)



**Careful:** Spark matches the schema to the columns **by position, not by name**. List every column, in the same order as in the file.
Here we forgot `artist_id`, so every column after `title` gets the wrong data — and `duration_ms` becomes null (`"pop"` is not a number):

In [6]:
wrong = spark.read.csv(
    str(DATA / "reference" / "tracks.csv"),
    header=True,
    schema="track_id STRING, title STRING, genre STRING, duration_ms INT",
)
wrong.show(3)

+--------+----------------+-----+-----------+
|track_id|           title|genre|duration_ms|
+--------+----------------+-----+-----------+
|   T0001|Electric Mirrors|  A01|       NULL|
|   T0002|Electric Skyline|  A01|       NULL|
|   T0003|Endless Promises|  A01|       NULL|
+--------+----------------+-----+-----------+
only showing top 3 rows


### Exercise 1 (Core) — Load the track catalogue
Read `DATA / "reference" / "tracks.csv"` into `tracks` with an explicit schema:

`track_id STRING, title STRING, artist_id STRING, genre STRING, duration_ms INT, release_year INT, explicit BOOLEAN`

Then show 5 rows and print the schema and the number of rows.

In [9]:
# YOUR CODE HERE
tracks = spark.read.csv(
    str(DATA / "reference" / "tracks.csv"),
    header = True,
    schema='track_id STRING, title STRING, artist_id STRING, genre STRING, duration_ms INT, release_year INT, explicit BOOLEAN')
tracks.show(5)
tracks.printSchema()
print(tracks.count(), "tracks")  # expected: 186 tracks

+--------+----------------+---------+-----+-----------+------------+--------+
|track_id|           title|artist_id|genre|duration_ms|release_year|explicit|
+--------+----------------+---------+-----+-----------+------------+--------+
|   T0001|Electric Mirrors|      A01|  pop|     237000|        2021|   false|
|   T0002|Electric Skyline|      A01|  pop|     200000|        2014|   false|
|   T0003|Endless Promises|      A01|  pop|     198000|        2022|   false|
|   T0004|   Golden Dreams|      A01|  pop|     193000|        2008|   false|
|   T0005|    Little Ocean|      A01|  pop|     186000|        2020|   false|
+--------+----------------+---------+-----+-----------+------------+--------+
only showing top 5 rows
root
 |-- track_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- artist_id: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- explicit: boolean (n

---
## Part 2 — Filter, sort, distinct

Last session you met `select` and `withColumn`. Three more transformations, one at a time.

### Keeping rows: `filter`
`filter` keeps the rows where the condition is true — like an `if` inside a loop in session 2. Note the **`==`** (compare), not `=`:

In [10]:
rock = tracks.filter(F.col("genre") == "rock")
rock.show(5)
print(rock.count(), "rock tracks")

+--------+---------------+---------+-----+-----------+------------+--------+
|track_id|          title|artist_id|genre|duration_ms|release_year|explicit|
+--------+---------------+---------+-----+-----------+------------+--------+
|   T0134|Electric Lights|      A15| rock|     305000|        2007|   false|
|   T0135| Golden Signals|      A15| rock|     228000|        2024|   false|
|   T0136| Hidden Skyline|      A15| rock|     277000|        2015|   false|
|   T0137|    Neon Hearts|      A15| rock|     235000|        2012|   false|
|   T0138| Silent Shadows|      A15| rock|     299000|        2025|   false|
+--------+---------------+---------+-----+-----------+------------+--------+
only showing top 5 rows
36 rock tracks


### Sorting: `orderBy`
`orderBy("column")` sorts from small to large. For large to small, use `F.desc("column")`:

In [11]:
tracks.orderBy("release_year").select("title", "release_year").show(3)
tracks.orderBy(F.desc("release_year")).select("title", "release_year").show(3)

+--------------+------------+
|         title|release_year|
+--------------+------------+
|Endless Dreams|        1976|
|   Summer Fire|        1976|
|    Wild Stars|        1979|
+--------------+------------+
only showing top 3 rows
+---------------+------------+
|          title|release_year|
+---------------+------------+
|Endless Highway|        2026|
|     Faded City|        2026|
|    Faded Stars|        2026|
+---------------+------------+
only showing top 3 rows


### Unique values: `distinct`
`distinct()` removes repeated rows — like `set(...)` in Python. Use it after `select` to see the different values of a column:

In [12]:
tracks.select("genre").distinct().show()

+----------+
|     genre|
+----------+
| classical|
|       pop|
|     latin|
|   hip-hop|
|electronic|
|      rock|
+----------+



### Putting it together
Every transformation returns a **new** DataFrame, so we can chain them, one step per line:

| Python (session 2) | Spark |
|---|---|
| pick some keys of each dict | `df.select("a", "b")` |
| add a key to each dict | `df.withColumn("minutes", F.col("seconds") / 60)` |
| `if` inside a loop to keep rows | `df.filter(F.col("seconds") >= 30)` |
| `sorted(...)` | `df.orderBy(F.desc("minutes"))` |
| `set(...)` | `df.select("device").distinct()` |

In [13]:
(
    tracks
    .select("title", "genre", "duration_ms")
    .withColumn("minutes", F.round(F.col("duration_ms") / 60000, 2))
    .filter(F.col("genre") == "rock")
    .orderBy(F.desc("minutes"))
    .show(5)
)

+--------------+-----+-----------+-------+
|         title|genre|duration_ms|minutes|
+--------------+-----+-----------+-------+
|Endless Dreams| rock|     330000|    5.5|
|     Wild City| rock|     328000|   5.47|
|   Wild Hearts| rock|     327000|   5.45|
|  Hidden River| rock|     320000|   5.33|
|Summer Thunder| rock|     313000|   5.22|
+--------------+-----+-----------+-------+
only showing top 5 rows


Combining conditions: use `&` (and), `|` (or), `~` (not), and put **each condition in parentheses**:

In [14]:
recent_pop = tracks.filter((F.col("genre") == "pop") & (F.col("release_year") >= 2020))
print(recent_pop.count())

not_pop = tracks.filter(~F.col("genre").isin("pop", "rock"))
not_pop.select("genre").distinct().show()

27
+----------+
|     genre|
+----------+
| classical|
|     latin|
|   hip-hop|
|electronic|
+----------+



### Exercise 2 (Core) — Long tracks
Build `long_tracks`: tracks longer than **5 minutes**, with columns `title`, `genre`, `minutes` (rounded to 1 decimal),
sorted longest first.

In [20]:
# YOUR CODE HERE
# tracks.show()
long_tracks = (
    tracks
    .withColumn('minutes', F.round(F.col("duration_ms") / 60000, 2))
    .select('title', 'genre', 'minutes')
    .filter((F.col('duration_ms') > 300000))
    .orderBy(F.desc("minutes"))
)


long_tracks.show(5)
print(long_tracks.count(), "long tracks")

+----------------+---------+-------+
|           title|    genre|minutes|
+----------------+---------+-------+
|    Cosmic River|classical|   8.72|
|Electric Skyline|classical|   8.42|
|  Broken Thunder|classical|   7.65|
| Burning Skyline|classical|   7.33|
|     Velvet Fire|classical|   6.52|
+----------------+---------+-------+
only showing top 5 rows
21 long tracks


### Exercise 3 (Core) — Explicit hip-hop
1. How many **explicit hip-hop** tracks released in **2015 or later** are there? Store the number in `n_explicit`.
2. Show the distinct `release_year` values of those tracks, sorted.

In [31]:
# YOUR CODE HERE
explicit_hiphop = tracks.filter(
     (F.col("genre") == "hip-hop")
    #   i marrim me zhaner hiphop

    & (F.col("explicit") == True)
    # dhe qe jane explicit

    & (F.col("release_year") >= 2015)
    # dhe 2025 ose ma vone te publikuara
  )

# e ruajme numrin sa jane
n_explicit = explicit_hiphop.count()
print(n_explicit)

explicit_hiphop.select("release_year").distinct().orderBy("release_year").show()


12
+------------+
|release_year|
+------------+
|        2015|
|        2017|
|        2019|
|        2020|
|        2022|
|        2024|
|        2025|
|        2026|
+------------+



**Stretch:** add a column `decade` (e.g. `1990`, `2010`) using `F.floor(F.col("release_year") / 10) * 10`, then count tracks per decade with `.groupBy("decade").count()`.

In [ ]:
# YOUR CODE HERE

---
## Part 3 — Lazy evaluation

Spark has two kinds of operations:

- **Transformations** (`select`, `filter`, `withColumn`, `orderBy`, ...) only *describe* what to do. They return instantly.
- **Actions** (`show`, `count`, `collect`, writing files, ...) make Spark actually *do* the work.

Watch the timings:

In [32]:
import time

start = time.perf_counter()
big = spark.range(30_000_000).withColumn("squared", F.col("id") * F.col("id")).filter(F.col("id") % 7 == 0)
print(f"transformations: {time.perf_counter() - start:.3f} s")  # nothing happened yet!

start = time.perf_counter()
print(big.count())
print(f"action:          {time.perf_counter() - start:.3f} s")  # now Spark works

transformations: 0.051 s
4285715
action:          2.305 s


Why lazy? Because Spark sees your **whole** plan before it starts, it can optimise it: skip columns you never use,
filter early, and split the work across executors. You can look at the plan with `explain()`:

In [33]:
big.explain()

== Physical Plan ==
*(1) Project [id#779L, (id#779L * id#779L) AS squared#780L]
+- *(1) Filter ((id#779L % 7) = 0)
   +- *(1) Range (0, 30000000, step=1, splits=2)




> **Under the hood:** read the plan from the bottom up. `Range` creates the numbers, `Filter` and `Project` (= select) run inside the
> same stage on every partition in parallel. You will read many of these plans in the distributed processing course.

---
## Part 4 — Investigating data quality with Spark

Back to the messy log. In session 2 we needed loops and sets to find problems. In Spark, each question is one line.

In [34]:
log.select("device").distinct().show()

print(log.filter(F.col("country").isNull()).count(), "rows without a country")

+-------------+
|       device|
+-------------+
|       MOBILE|
|smart_speaker|
|SMART_SPEAKER|
|      DESKTOP|
|Smart_speaker|
|          web|
|          WEB|
|smart speaker|
|       Mobile|
|      Desktop|
|       mobile|
|      desktop|
|          Web|
+-------------+

48 rows without a country


Note: Spark reads **empty CSV fields as `null`** (missing). Test for it with `.isNull()` / `.isNotNull()` — never with `== None`.

`groupBy(...).count()` counts rows per value:

In [ ]:
log.groupBy("country").count().orderBy(F.desc("count")).show()

### Exercise 4 (Core) — Find the problems
Answer with Spark:
1. `n_rows` and `n_distinct_rows`: row count before and after `.distinct()`. How many exact duplicates are there?
2. `n_missing_duration`: rows where `duration` is null.
3. Show how many rows have each distinct spelling of `artist` for **Eminem** (hint: `F.lower(F.trim(F.col("artist"))) == "eminem"`).

In [52]:
# YOUR CODE HERE
log.show()
n_rows = log.count()
print(n_rows)

n_distinct_rows = log.distinct().count()
print(n_distinct_rows)

n_missing_duration = log.filter(F.col("duration").isNull()).count()



log.filter(F.lower(F.trim(F.col("artist"))) == "eminem").groupBy('artist').count().orderBy('artist').show()

print(n_rows, n_distinct_rows, n_rows - n_distinct_rows, n_missing_duration)  # expected: 625 600 25 26

+-------+-------------------+-----+---------------+-------------------+--------+-------+-------------+
|play_id|          played_at| user|         artist|              track|duration|country|       device|
+-------+-------------------+-----+---------------+-------------------+--------+-------+-------------+
|      1|   01/09/2026 23:56|U0293|  Calvin Harris|  Little Promises  |      47|     DE|       mobile|
|      2|   03/09/2026 21:21|U0025|  Billie Eilish|    Little Lights  |    1:25|     AT|       Mobile|
|      3|2026-09-02 12:52:37|U0075|         EMINEM| Northern Thunder  |      86|     SK|       mobile|
|      4|   02/09/2026 18:38|U0046|         Eminem|     Broken Shadows|    0:17|     SK|       mobile|
|      5|2026-09-06 11:03:45|U0396| Kendrick Lamar|    electric lights|      87|     AT|       Mobile|
|      6|2026-09-01 16:43:09|U0006|     Ed Sheeran|       Paper Dreams|     146|     HU|       mobile|
|      7|  1 Sep 2026, 04:31|U0676|      Bad Bunny|    Endless Thunder|  

### Exercise 5 (Core) — Duration formats
`rlike` tests a column against a **regular expression** (a text pattern):
- `"^[0-9]+$"` — only digits, like `225`
- `"^[0-9]+:[0-9]{2}$"` — like `3:45`
- `"^[0-9]+s$"` — like `225s`

Add a column `duration_format` with value `"seconds"`, `"m:ss"`, `"with s"` or `"other"`, then count rows per format.
Use `F.when(condition, value).when(...).otherwise(value)` — Spark's `if / elif / else`.

In [62]:
# YOUR CODE HERE
log.show()
formats = log.withColumn('duration_format',
                         F.when(F.col('duration').rlike("^[0-9]+$"), 'seconds')
                         .when(F.col('duration').rlike("^[0-9]+:[0-9]{2}$"), 'm:ss')
                         .when(F.col('duration').rlike("^[0-9]+s$"), 'with s')
                         .otherwise('other')
                         )

formats.groupBy('duration_format').count().show()

+-------+-------------------+-----+---------------+-------------------+--------+-------+-------------+
|play_id|          played_at| user|         artist|              track|duration|country|       device|
+-------+-------------------+-----+---------------+-------------------+--------+-------+-------------+
|      1|   01/09/2026 23:56|U0293|  Calvin Harris|  Little Promises  |      47|     DE|       mobile|
|      2|   03/09/2026 21:21|U0025|  Billie Eilish|    Little Lights  |    1:25|     AT|       Mobile|
|      3|2026-09-02 12:52:37|U0075|         EMINEM| Northern Thunder  |      86|     SK|       mobile|
|      4|   02/09/2026 18:38|U0046|         Eminem|     Broken Shadows|    0:17|     SK|       mobile|
|      5|2026-09-06 11:03:45|U0396| Kendrick Lamar|    electric lights|      87|     AT|       Mobile|
|      6|2026-09-01 16:43:09|U0006|     Ed Sheeran|       Paper Dreams|     146|     HU|       mobile|
|      7|  1 Sep 2026, 04:31|U0676|      Bad Bunny|    Endless Thunder|  

**Stretch:** which `played_at` formats exist? Use `rlike` to classify ISO (`^[0-9]{4}-`), slash (`/`) and "text month" (`[A-Za-z]`) timestamps and count them.

In [68]:
# YOUR CODE HERE
log.withColumn(
    'ts_format',
    F.when(F.col('played_at').rlike("^[0-9]{4}-"), 'ISO')
    .when(F.col('played_at').contains("/"), 'slash')
    .when(F.col('played_at').rlike("[A-Za-z]"), 'text month')
    .otherwise('other')
).groupBy('ts_format').count().show()

+----------+-----+
| ts_format|count|
+----------+-----+
|     slash|  169|
|       ISO|  331|
|text month|  125|
+----------+-----+



---
## Wrap-up

| Concept | Example |
|---|---|
| read CSV with a schema | `spark.read.csv(path, header=True, schema="a STRING, b INT")` — columns by position! |
| keep rows | `df.filter(F.col("genre") == "rock")` |
| combine conditions | `(cond1) & (cond2)`, `\|`, `~`, `F.col("x").isin(...)` |
| sort | `df.orderBy("a")`, `df.orderBy(F.desc("a"))` |
| unique values | `df.select("a").distinct()` |
| count per value | `df.groupBy("a").count()` |
| if / elif / else | `F.when(cond, value).when(...).otherwise(value)` |
| lazy | transformations plan, actions run |
| missing values | `F.col("x").isNull()` |

Next session we **fix** all the problems we found, combine tables with **joins** and save results as **Parquet**.

**Homework:** `homework/hw04_spark_basics.ipynb`